<a href="https://colab.research.google.com/github/christiejibaraki/CUREBench/blob/finetune-gpt-oss-20b/notebooks/4-explore_finetuned_gpt_oss_20b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Run finetuned gpt-oss-20b

In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [2]:
import os
import sys
import os.path

In [3]:
os.environ['BNB_CUDA_VERSION'] = '125'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

## Setup
- Clone forked CUREBench repo onto local `content` folder (this is not persistent)
- Create virtual environment and install packages
  - Package installation takes about **5 min** ⏰
- Add virtual environment's site-packages to notebook's system path

In [4]:
!git clone -b finetune-gpt-oss-20b https://github.com/christiejibaraki/CUREBench.git

Cloning into 'CUREBench'...
remote: Enumerating objects: 299, done.
remote: Counting objects: 100% (210/210), done.
remote: Compressing objects: 100% (163/163), done.
Receiving objects: 100% (299/299), 3.20 MiB | 25.57 MiB/s, done.
remote: Total 299 (delta 130), reused 90 (delta 47), pack-reused 89 (from 2)
Resolving deltas: 100% (159/159), done.


In [1]:
%cd CUREBench

/content/CUREBench


In [2]:
!git pull

remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 439 bytes | 146.00 KiB/s, done.
From https://github.com/christiejibaraki/CUREBench
   ccbbc51..d65fe65  finetune-gpt-oss-20b -> origin/finetune-gpt-oss-20b
Updating ccbbc51..d65fe65
Fast-forward
 core/eval_framework.py | 9 +++++----
 1 file changed, 5 insertions(+), 4 deletions(-)


In [3]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [4]:
%%capture
!uv pip install --force-reinstall --no-deps git+https://github.com/unslothai/unsloth-zoo
!uv pip install --force-reinstall --no-deps git+https://github.com/unslothai/unsloth

### manually load finetuned model

In [4]:
from unsloth import FastLanguageModel

max_seq_length = 1024
dtype = None

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = dtype, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.1: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
model.load_adapter("cibaraki/medical-reasoning-gpt-oss-20b")

In [6]:
messages = [
    {"role": "system", "content": "You are a medical expert for drug decision-making and treatment planning. First, determine if the question is multiple-choice (MC) or open-ended (OE). For MC questions, your final response must be ONLY the correct LETTER. For OE questions, provide a succinct, single sentence response."},
    {"role": "user", "content": 'What precaution should be taken for patients with a history of allergic disorders before administering Gadavist?'},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

output_ids = model.generate(**inputs, max_new_tokens = 512)

In [10]:
response = tokenizer.decode(output_ids[0], skip_special_tokens=False)

In [11]:
response

"<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.\nKnowledge cutoff: 2024-06\nCurrent date: 2025-11-04\n\nReasoning: low\n\n# Valid channels: analysis, commentary, final. Channel must be included for every message.\nCalls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions\n\nYou are a medical expert for drug decision-making and treatment planning. First, determine if the question is multiple-choice (MC) or open-ended (OE). For MC questions, your final response must be ONLY the correct LETTER. For OE questions, provide a succinct, single sentence response.<|end|><|start|>user<|message|>What precaution should be taken for patients with a history of allergic disorders before administering Gadavist?<|end|><|start|>assistant<|channel|>analysis<|message|>The question is open-ended. Provide a concise answer.<|end|><|start|>assistant<|channel|>final<|message|>Patients with a history of allergic di

In [21]:
import re
import json # Used for structuring the final output

full_response = response

## regex
# pattern to find all all assistant messages
ASSISTANT_MESSAGE_PATTERN = re.compile(
    r'<\|start\|>assistant(?P<header>.+?)<\|message\|>(?P<content>.*?)(?=<\|end\|>|<\|start\|>|<\|return\|>|$)',
    re.DOTALL | re.IGNORECASE
)

# pattern to extract the channel from the header
CHANNEL_PATTERN = re.compile(r'<\|channel\|>(?P<channel>\w+)')


reasoning_trace = []
final_response = "ERROR: Final response not found."

# Find all assistant messages in the full response
assistant_messages = ASSISTANT_MESSAGE_PATTERN.findall(full_response)

for header, content in assistant_messages:
    # Extract the channel(s) from the header
    channel_matches = CHANNEL_PATTERN.findall(header)

    message_dict = {
        "role": "assistant",
        "channel": channel_matches[-1] if channel_matches else "unknown",
        "content": content.strip()
    }

    # Add to the reasoning trace
    reasoning_trace.append(message_dict)

    # Check if this is the final message
    if message_dict['channel'] == 'final':
        final_response = message_dict['content']


# --- Results ---

print("="*60)
print(f"Final Answer (from 'final' channel): **{final_response}**")
print("="*60)
print("Reasoning Trace (List[Dict] format):")
print(json.dumps(reasoning_trace, indent=4))
print("="*60)

Final Answer (from 'final' channel): **Patients with a history of allergic disorders should use Gadolinium-based contrast agents like Gadavist with caution due to the risk of allergic reactions; premedication protocols or alternative imaging options may be considered.**
Reasoning Trace (List[Dict] format):
[
    {
        "role": "assistant",
        "channel": "analysis",
        "content": "The question is open-ended. Provide a concise answer."
    },
    {
        "role": "assistant",
        "channel": "final",
        "content": "Patients with a history of allergic disorders should use Gadolinium-based contrast agents like Gadavist with caution due to the risk of allergic reactions; premedication protocols or alternative imaging options may be considered."
    }
]


### load model from eval_framework

In [4]:
from core.eval_framework import UnslothGPTOSS20BModel

model_name = "unsloth/gpt-oss-20b"
developer_instructions = "You are a medical expert for drug decision-making and treatment planning. During your analysis, determine if the question is multiple-choice (MC) or open-ended (OE). For MC questions, your final response must be ONLY the correct LETTER. For OE questions, provide a succinct, single sentence response."
lora_adapters = "cibaraki/medical-reasoning-gpt-oss-20b"

model = UnslothGPTOSS20BModel(model_name=model_name,
                              developer_instructions=developer_instructions,
                              lora_adapters=lora_adapters)

In [5]:
model.load()

/content/CUREBench/core/eval_framework.py:161: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.1: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [9]:
prompt = "Which of the following is NOT an indicated use for 'up and up ibuprofen'?\nA: Relief of occasional sleeplessness\nB: Relief of minor aches and pains\nC: Treatment of chronic pain conditions\nD: Helping users stay asleep"
final_response, reasoning_trace = model.inference(prompt=prompt,
                max_tokens=512)

In [10]:
final_response

'C'

In [11]:
reasoning_trace

[{'role': 'assistant',
  'channel': 'analysis',
  'content': "The question asks which option is NOT an indicated use for 'up and up ibuprofen'. Options discuss various uses: A: Relieving occasional sleeplessness, B: relief of minor aches and pains, C: treatment of chronic pain conditions, D: helping users stay asleep. The correct answer likely states that ibuprofen is not used for helping with sleep or treating chronic pain? Actually, ibuprofen is used for minor aches and pains but not for chronic pain as a long-term solution. So answer C or D? Typically, NSAIDs are not used for chronic pain treatment as first line. They can treat acute pain. So choice C seems inappropriate.\n\nThus answer: C."},
 {'role': 'assistant', 'channel': 'final', 'content': 'C'}]

### open-ended multiple choice

In [13]:
prompt = "What should patients do if they experience severe allergic reactions during or after receiving fosaprepitant for injection?"
final_response, reasoning_trace = model.inference(prompt=prompt,
                max_tokens=512)

In [14]:
final_response

'If severe allergic reactions occur during or after injection of fosaprepitant, patients should promptly seek emergency medical attention, contact emergency services, or go to the nearest emergency department for immediate evaluation and treatment.'

In [15]:
reasoning_trace

[{'role': 'assistant',
  'channel': 'analysis',
  'content': "The question is open-ended: it asks what patients should do if they experience severe allergic reactions during or after receiving fosaprepitant for injection. It's not multiple choice."},
 {'role': 'assistant',
  'channel': 'final',
  'content': 'If severe allergic reactions occur during or after injection of fosaprepitant, patients should promptly seek emergency medical attention, contact emergency services, or go to the nearest emergency department for immediate evaluation and treatment.'}]

In [17]:
meta_question = "The following is a multiple choice question about medicine and the agent's open-ended answer to the question. Convert the agent's answer to the final answer format using the corresponding option label, e.g., 'A', 'B', 'C', 'D', 'E' or 'None'. \n\nQuestion: What should patients do if they experience severe allergic reactions during or after receiving fosaprepitant for injection?\nA: Wait for the symptoms to resolve on their own.\nB: Inform their healthcare provider immediately and seek emergency medical care.\nC: Stop chemotherapy treatment permanently.\nD: Take over-the-counter antihistamines.\n\n"
meta_prompt = f"{meta_question}Agent's answer: {final_response}\n\nMulti-choice answer:"
meta_prompt

"The following is a multiple choice question about medicine and the agent's open-ended answer to the question. Convert the agent's answer to the final answer format using the corresponding option label, e.g., 'A', 'B', 'C', 'D', 'E' or 'None'. \n\nQuestion: What should patients do if they experience severe allergic reactions during or after receiving fosaprepitant for injection?\nA: Wait for the symptoms to resolve on their own.\nB: Inform their healthcare provider immediately and seek emergency medical care.\nC: Stop chemotherapy treatment permanently.\nD: Take over-the-counter antihistamines.\n\nAgent's answer: If severe allergic reactions occur during or after injection of fosaprepitant, patients should promptly seek emergency medical attention, contact emergency services, or go to the nearest emergency department for immediate evaluation and treatment.\n\nMulti-choice answer:"

In [18]:
meta_response, meta_trace = model.inference(prompt=meta_prompt,
                max_tokens=512)

In [19]:
meta_response

'B'

In [20]:
meta_trace

[{'role': 'assistant',
  'channel': 'analysis',
  'content': 'We need to convert to the correct option label. The correct option: "Inform their healthcare provider immediately and seek emergency medical care." That\'s option B.'},
 {'role': 'assistant', 'channel': 'final', 'content': 'B'}]

### read config

In [5]:
import json

In [6]:
config_path= "metadata_config_val.json"
config = json.load(open(config_path, 'r')) if config_path else {}
if 'dataset' in config:
    dataset_config = config['dataset']
    dataset_name = dataset_config.get('dataset_name', 'treatment')
print(f"\nconfig file: {config_path}\ncontents:\n{dataset_config}")
dataset_path = dataset_config.get("dataset_path")


config file: metadata_config_val.json
contents:
{'dataset_name': 'cure_bench_phase1_val', 'dataset_path': 'resources/curebench_valset_pharse1.jsonl', 'description': 'CureBench 2025 val questions'}


In [7]:
config

{'metadata': {'model_name': 'unsloth/gpt-oss-20b',
  'model_type': 'open weight model',
  'track': 'internal_reasoning',
  'base_model_type': 'open weight model',
  'base_model_name': 'gpt-oss-20b',
  'dataset': 'cure_bench_phase1_val',
  'additional_info': 'Submission using configuration file'},
 'dataset': {'dataset_name': 'cure_bench_phase1_val',
  'dataset_path': 'resources/curebench_valset_pharse1.jsonl',
  'description': 'CureBench 2025 val questions'},
 'output_dir': 'competition_test_results'}

### load model with competition kit

In [8]:
from core.eval_framework import CompetitionKit, load_and_merge_config, create_metadata_parser

In [9]:
model_name = config['metadata']['model_name']
model_class = model_name # updated metadata parser to parse model_class

In [10]:
print(f"model name: {model_name}")
print(f"model class: {model_class}")

model name: unsloth/gpt-oss-20b
model class: unsloth/gpt-oss-20b


In [11]:
kit = CompetitionKit(config_path=config_path)

In [12]:
print(f"Loading model: {model_name}")
kit.load_model(model_name, model_class)

Loading model: unsloth/gpt-oss-20b


/content/CUREBench/core/eval_framework.py:165: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.1: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [13]:
kit.model.developer_instructions

'You are a medical expert for drug decision-making and treatment planning. During your analysis, determine if the question is multiple-choice (MC) or open-ended (OE). For MC questions, your final response must be ONLY the correct LETTER. For OE questions, provide a succinct, single sentence response.'

In [14]:
kit.list_datasets()

Available Datasets:
--------------------------------------------------
  cure_bench_phase1_val: CureBench 2025 val questions


### run model with evaluate
- takes about 23 seconds to do one example on L4 gpu

In [15]:
subset_size = 3
print(f"Running evaluation on dataset: {dataset_name} (subset-size={subset_size})")
results = kit.evaluate(dataset_name, subset_size=subset_size)

Running evaluation on dataset: cure_bench_phase1_val (subset-size=3)
dataset_path: resources/curebench_valset_pharse1.jsonl
CureBenchDataset initialized with 459 examples


Evaluating: 100%|██████████| 3/3 [04:42<00:00, 94.04s/it]


In [16]:
results

EvaluationResult(dataset_name='cure_bench_phase1_val', model_name='unsloth/gpt-oss-20b', accuracy=1.0, correct_predictions=3, total_examples=3, predictions=[{'choice': 'A', 'open_ended_answer': 'A'}, {'choice': 'B', 'open_ended_answer': 'If patients experience severe allergic reactions during or after receiving fosaprepitant for injection, they should seek immediate medical attention by contacting emergency services or going to the nearest emergency department.'}, {'choice': 'B', 'open_ended_answer': 'If the dose indicator on Stiolto Respimat reaches 0, you should replace the device with a new one.'}], reasoning_traces=[[{'role': 'assistant', 'channel': 'analysis', 'content': 'This is a multiple-choice question asking to identify the drug brand name associated with the treatment of acne. Salicylic acid is often used for acne treatment. Therefore, the correct answer is A.'}, {'role': 'assistant', 'channel': 'final', 'content': 'A'}], [{'role': 'assistant', 'channel': 'analysis', 'conten